In [12]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

#assume defauly dir in /255-AI/ dir
os.chdir(Path(globals()['_dh'][0]).parent.parent)
print(os.getcwd())

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)

RAW        = Path('data/raw')
PROCESSED  = Path('data/processed')

BLS_FILES  = {
    2019: RAW / 'bls/oesm19nat.xlsx',
    2022: RAW / 'bls/oesm22nat.xlsx',
    2024: RAW / 'bls/oesm24nat.xlsx',
}
ONET_ABILITIES   = RAW / 'onet/Abilities.xlsx'
ONET_ACTIVITIES  = RAW / 'onet/Work Activities.xlsx'
ONET_OCCUPATIONS = RAW / 'onet/Occupation Data.xlsx'

FELTEN_FILE = RAW / 'felten/AIOE_DataAppendix.xlsx'

list(Path('data/').iterdir())

xls = pd.ExcelFile(Path('data/raw/felten/AIOE_DataAppendix.xlsx'))
print(xls.sheet_names)

C:\Users\pchau\Documents\Masters Y2, S2\CMPE-255\Project\Branch\255-AI
['Index', 'Appendix A', 'Appendix B', 'Appendix C', 'Appendix D', 'Appendix E']


In [13]:
#standardise SOC codes to format XX-XXXX.
def normalize_soc(series):
    s = series.astype(str).str.strip()
    s = s.str.replace(r'\.0+$', '', regex=True)
    mask_no_hyphen = ~s.str.contains('-')
    s.loc[mask_no_hyphen] = (
        s.loc[mask_no_hyphen].str.zfill(6)
         .str[:2] + '-' + s.loc[mask_no_hyphen].str.zfill(6).str[2:]
    )
    return s

#lowercase and strip all column names
def col_lower(df):
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    return df

#change to numeric, treating BLS suppression markers ('**', '#') as NaN
def to_numeric_safe(series):
    return pd.to_numeric(series.astype(str).str.replace(r'[#*,]', '', regex=True),
                         errors='coerce')


print('Helpers')

Helpers


In [14]:
#BLS OEWS columns of interest
BLS_KEEP = [
    'occ_code', 'occ_title', 'o_group',
    'tot_emp', 'emp_prse',
    'a_mean', 'a_median', 'a_pct10', 'a_pct25', 'a_pct75', 'a_pct90',
    'h_mean', 'h_median',
]

#load one BLS OEWS national file and return detail-level occupations only
def load_bls_vintage(path, year):
    xls = pd.ExcelFile(path)
    #prefer a sheet whose name contains 'national' (case-insensitive)
    target = next(
        (s for s in xls.sheet_names if 'national' in s.lower()),
        xls.sheet_names[0]
    )
    df = xls.parse(target, dtype=str)
    df = col_lower(df)

    #keep only columns that exist in this vintage
    keep = [c for c in BLS_KEEP if c in df.columns]
    df = df[keep].copy()

    #restrict to 6-digit (detailed) occupations; exclude aggregates & totals
    df = df[df['o_group'].str.strip().str.upper() == 'DETAILED'].copy()
    df['soc6'] = normalize_soc(df['occ_code'])

    #change wage / employment columns
    numeric_cols = [c for c in keep if c not in ('occ_code', 'occ_title', 'o_group')]
    for c in numeric_cols:
        df[c] = to_numeric_safe(df[c])

    #tag year
    df['bls_year'] = year

    df = df.drop(columns=['occ_code', 'o_group'])
    return df


bls_frames = {yr: load_bls_vintage(path, yr) for yr, path in BLS_FILES.items()}

for yr, df in bls_frames.items():
    print(f'BLS {yr}: {len(df):,} detailed occupations | columns: {list(df.columns)}')

BLS 2019: 789 detailed occupations | columns: ['occ_title', 'tot_emp', 'emp_prse', 'a_mean', 'a_median', 'a_pct10', 'a_pct25', 'a_pct75', 'a_pct90', 'h_mean', 'h_median', 'soc6', 'bls_year']
BLS 2022: 830 detailed occupations | columns: ['occ_title', 'tot_emp', 'emp_prse', 'a_mean', 'a_median', 'a_pct10', 'a_pct25', 'a_pct75', 'a_pct90', 'h_mean', 'h_median', 'soc6', 'bls_year']
BLS 2024: 831 detailed occupations | columns: ['occ_title', 'tot_emp', 'emp_prse', 'a_mean', 'a_median', 'a_pct10', 'a_pct25', 'a_pct75', 'a_pct90', 'h_mean', 'h_median', 'soc6', 'bls_year']


In [15]:
#pivot to wide format: one row per SOC, wage columns suffixed by year
bls_wide_frames = []

for yr, df in bls_frames.items():
    wage_cols = [c for c in df.columns if c not in ('soc6', 'occ_title', 'bls_year')]
    renamed = {c: f'{c}_{yr}' for c in wage_cols}
    tmp = df.rename(columns=renamed).drop(columns='bls_year')
    bls_wide_frames.append(tmp.set_index('soc6'))

#outer join within BLS so we keep all SOC codes present in any vintage,
#then later the master merge will inner-join with O*NET & Felten
bls = bls_wide_frames[0].join(bls_wide_frames[1], how='outer', rsuffix='_dup')
bls = bls.join(bls_wide_frames[2], how='outer', rsuffix='_dup2')

#resolve duplicate occ_title columns, keep the most recent non-null
title_cols = [c for c in bls.columns if c.startswith('occ_title')]
bls['occ_title'] = bls[title_cols].bfill(axis=1).iloc[:, 0]
bls = bls.drop(columns=[c for c in title_cols if c != 'occ_title'])
bls = bls.drop(columns=[c for c in bls.columns if '_dup' in c])

bls = bls.reset_index()
print(f'BLS wide: {len(bls):,} SOC codes | {bls.shape[1]} columns')
bls.head(3)

BLS wide: 861 SOC codes | 32 columns


,soc6,occ_title,tot_emp_2019,emp_prse_2019,a_mean_2019,a_median_2019,a_pct10_2019,a_pct25_2019,a_pct75_2019,a_pct90_2019,h_mean_2019,h_median_2019,tot_emp_2022,emp_prse_2022,a_mean_2022,a_median_2022,a_pct10_2022,a_pct25_2022,a_pct75_2022,a_pct90_2022,h_mean_2022,h_median_2022,tot_emp_2024,emp_prse_2024,a_mean_2024,a_median_2024,a_pct10_2024,a_pct25_2024,a_pct75_2024,a_pct90_2024,h_mean_2024,h_median_2024
0,11-1011,Chief Executives,"205,890.0000",0.8000,"193,850.0000","184,460.0000","62,290.0000","112,790.0000",NaN,NaN,93.2000,88.6800,"199,240.0000",0.9000,"246,440.0000","189,520.0000","74,920.0000","122,480.0000",NaN,NaN,118.4800,91.1200,"211,850.0000",1.2000,"262,930.0000","206,420.0000","73,710.0000","126,080.0000",NaN,NaN,126.4100,99.2400
1,11-1021,General and Operations Managers,"2,400,280.0000",0.3000,"123,030.0000","100,780.0000","45,050.0000","65,660.0000","157,430.0000",NaN,59.1500,48.4500,"3,376,680.0000",0.3000,"122,860.0000","98,100.0000","43,470.0000","62,070.0000","154,560.0000","221,270.0000",59.0700,47.1600,"3,584,420.0000",0.4000,"133,120.0000","102,950.0000","47,420.0000","67,160.0000","164,130.0000",NaN,64.0000,49.5000
2,11-1031,Legislators,"52,280.0000",2.2000,"49,440.0000","29,270.0000","17,690.0000","19,070.0000","75,520.0000","100,470.0000",NaN,NaN,"42,890.0000",2.1000,"71,100.0000","48,090.0000","20,970.0000","28,690.0000","94,030.0000","149,710.0000",NaN,NaN,"26,510.0000",3.9000,"67,390.0000","44,810.0000","20,380.0000","29,120.0000","80,350.0000","137,820.0000",NaN,NaN


In [16]:
#occupation data
onet_occ = col_lower(pd.read_excel(ONET_OCCUPATIONS, dtype=str))
onet_occ = onet_occ.rename(columns={'o*net-soc_code': 'onet_soc',
                                     'onetsoc_code': 'onet_soc'})
#accept column name present
soc_col = next(c for c in onet_occ.columns if 'soc' in c or 'code' in c)
onet_occ = onet_occ.rename(columns={soc_col: 'onet_soc'})
onet_occ['soc6'] = normalize_soc(onet_occ['onet_soc'].str[:7])  # trim to 6-digit

keep_cols = ['soc6'] + [c for c in onet_occ.columns
                         if c not in ('onet_soc', 'soc6') and 'title' in c or 'description' in c]
onet_occ = onet_occ[list(dict.fromkeys(['soc6'] + keep_cols))].copy()
onet_occ.columns = ['soc6'] + [f'onet_{c}' for c in onet_occ.columns if c != 'soc6']

print(f'O*NET Occupation Data: {len(onet_occ):,} rows | columns: {list(onet_occ.columns)}')

O*NET Occupation Data: 1,016 rows | columns: ['soc6', 'onet_title', 'onet_description']


In [17]:
#load an O*NET element file (Abilities or Work Activities).
#pivot to wide format: one row per SOC, one column per element
#(using the Importance scale LV score as default).
def pivot_onet_ratings(path, label):
    df = col_lower(pd.read_excel(path, dtype=str))

    #identify key columns
    soc_col   = next(c for c in df.columns if 'soc' in c or 'code' in c)
    elem_col  = next((c for c in df.columns if 'element_name' in c or 'element name' in c), None)
    scale_col = next((c for c in df.columns if 'scale_id' in c or 'scale id' in c), None)
    data_col  = next((c for c in df.columns if c in ('data_value', 'data value')), None)

    if elem_col is None or scale_col is None or data_col is None:
        raise ValueError(f'Cannot identify required columns in {path}. Found: {list(df.columns)}')

    df['soc6'] = normalize_soc(df[soc_col].str[:7])
    df[data_col] = pd.to_numeric(df[data_col], errors='coerce')

    #importance scale
    scales_present = df[scale_col].unique()
    scale_pref = 'IM' if 'IM' in scales_present else ('LV' if 'LV' in scales_present else scales_present[0])
    sub = df[df[scale_col] == scale_pref].copy()

    #average
    sub = sub.groupby(['soc6', elem_col])[data_col].mean().reset_index()

    #pivot
    wide = sub.pivot(index='soc6', columns=elem_col, values=data_col)
    wide.columns = [f'{label}__{re.sub(r"[^a-z0-9]+", "_", c.lower().strip())}'
                    for c in wide.columns]
    wide = wide.reset_index()

    print(f'O*NET {label}: {len(wide):,} SOC codes | {wide.shape[1]-1} elements')
    return wide


onet_abilities   = pivot_onet_ratings(ONET_ABILITIES,  'ability')
onet_activities  = pivot_onet_ratings(ONET_ACTIVITIES, 'workact')

#merge onet tables
onet = onet_occ.merge(onet_abilities,  on='soc6', how='inner')
onet = onet.merge(onet_activities, on='soc6', how='inner')

print(f'O*NET merged: {len(onet):,} SOC codes | {onet.shape[1]} columns')
onet.head(3)

O*NET ability: 774 SOC codes | 52 elements
O*NET workact: 774 SOC codes | 41 elements
O*NET merged: 923 SOC codes | 96 columns


,soc6,onet_title,onet_description,ability__arm_hand_steadiness,ability__auditory_attention,ability__category_flexibility,ability__control_precision,ability__deductive_reasoning,ability__depth_perception,ability__dynamic_flexibility,ability__dynamic_strength,ability__explosive_strength,ability__extent_flexibility,ability__far_vision,ability__finger_dexterity,ability__flexibility_of_closure,ability__fluency_of_ideas,ability__glare_sensitivity,ability__gross_body_coordination,ability__gross_body_equilibrium,ability__hearing_sensitivity,ability__inductive_reasoning,ability__information_ordering,ability__manual_dexterity,ability__mathematical_reasoning,ability__memorization,ability__multilimb_coordination,ability__near_vision,ability__night_vision,ability__number_facility,...,workact__establishing_and_maintaining_interpersonal_relationships,workact__estimating_the_quantifiable_characteristics_of_products_events_or_information,workact__evaluating_information_to_determine_compliance_with_standards,workact__getting_information,workact__guiding_directing_and_motivating_subordinates,workact__handling_and_moving_objects,workact__identifying_objects_actions_and_events,workact__inspecting_equipment_structures_or_materials,workact__interpreting_the_meaning_of_information_for_others,workact__judging_the_qualities_of_objects_services_or_people,workact__making_decisions_and_solving_problems,workact__monitoring_processes_materials_or_surroundings,workact__monitoring_and_controlling_resources,workact__operating_vehicles_mechanized_devices_or_equipment,workact__organizing_planning_and_prioritizing_work,workact__performing_administrative_activities,workact__performing_general_physical_activities,workact__performing_for_or_working_directly_with_the_public,workact__processing_information,workact__providing_consultation_and_advice_to_others,workact__repairing_and_maintaining_electronic_equipment,workact__repairing_and_maintaining_mechanical_equipment,workact__resolving_conflicts_and_negotiating_with_others,workact__scheduling_work_and_activities,workact__selling_or_influencing_others,workact__staffing_organizational_units,workact__thinking_creatively,workact__training_and_teaching_others,workact__updating_and_using_relevant_knowledge,workact__working_with_computers
0,11-1011,Chief Executives,Determine and formulate policies and provide o...,1.1900,2.0600,3.3100,1.6250,4.0000,1.8750,1.0000,1.1250,1.0000,1.0000,2.9400,1.6250,3.0650,3.8800,1.1250,1.0000,1.0000,2.0600,3.9400,3.8100,1.0000,3.0000,2.6300,1.6250,3.3700,1.1250,3.0000,...,4.7350,3.4000,4.0800,4.6700,4.2600,1.6500,4.0400,2.3100,4.2950,4.0150,4.7400,3.8650,3.9950,2.1850,4.4500,3.6600,1.8850,3.7300,4.2100,3.9400,1.4650,1.3100,3.9550,3.7300,3.9600,3.5200,4.3250,3.8200,4.3050,4.1600
1,11-1011,Chief Sustainability Officers,"Communicate and coordinate with management, sh...",1.1900,2.0600,3.3100,1.6250,4.0000,1.8750,1.0000,1.1250,1.0000,1.0000,2.9400,1.6250,3.0650,3.8800,1.1250,1.0000,1.0000,2.0600,3.9400,3.8100,1.0000,3.0000,2.6300,1.6250,3.3700,1.1250,3.0000,...,4.7350,3.4000,4.0800,4.6700,4.2600,1.6500,4.0400,2.3100,4.2950,4.0150,4.7400,3.8650,3.9950,2.1850,4.4500,3.6600,1.8850,3.7300,4.2100,3.9400,1.4650,1.3100,3.9550,3.7300,3.9600,3.5200,4.3250,3.8200,4.3050,4.1600
2,11-1021,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",1.6200,2.1200,3.3800,1.1200,3.8800,1.8800,1.0000,1.1200,1.3800,1.2500,2.6200,1.5000,2.6200,3.2500,1.0000,1.5000,1.3800,1.7500,3.5000,3.5000,1.5000,2.8800,2.2500,1.5000,3.5000,1.1200,2.8800,...,4.2800,3.2300,3.5900,4.4200,4.2000,2.0000,4.2200,2.6200,3.7200,4.1000,4.3300,3.9100,3.8600,1.9000,4.2000,3.4000,2.1100,2.5400,4.0800,3.6600,1.8900,1.8300,3.8800,3.8100,3.5800,3.7000,3.7000,3.4100,3.8000,4.4600


In [18]:
#load all appendix sheets from AIOE_DataAppendix.xlsx
#sheets are outer-joined on soc6 so all occupations are kept
def load_felten_appendix(path):
    xls = pd.ExcelFile(path)

    #filter to the 5 appendix sheets
    appendix_sheets = [s for s in xls.sheet_names if 'appendix' in s.lower()]

    if not appendix_sheets:
        #fall back to all sheets if none are named 'appendix'
        appendix_sheets = xls.sheet_names
        print(f'Warning: no sheets with "appendix" in name found. Using all sheets: {appendix_sheets}')

    frames = []
    for sheet in appendix_sheets:
        df = col_lower(xls.parse(sheet, dtype=str))

        #find SOC column
        soc_col = next(
            (c for c in df.columns if 'soc' in c or 'occ_code' in c or 'code' in c),
            None
        )
        if soc_col is None:
            print(f'Skipping sheet "{sheet}": cannot find SOC column. Columns: {list(df.columns)}')
            continue

        df['soc6'] = normalize_soc(df[soc_col])

        score_cols = [c for c in df.columns if c not in (soc_col, 'soc6') and
                      not any(kw in c for kw in ('title', 'name', 'description'))]

        for c in score_cols:
            df[c] = pd.to_numeric(df[c], errors='coerce')

        #prefix columns with sheet name (ex. "Appendix A" to "aioe_appendix_a_")
        prefix = re.sub(r'[^a-z0-9]+', '_', sheet.lower().strip()).strip('_')
        rename_map = {c: f'aioe_{prefix}_{c}' for c in score_cols}
        df = df.rename(columns=rename_map)
        df = df[['soc6'] + list(rename_map.values())]

        #average duplicate SOC codes in the sheet
        df = df.groupby('soc6').mean(numeric_only=True).reset_index()

        print(f'  Sheet "{sheet}": {len(df):,} SOC codes | {df.shape[1]-1} score columns')
        frames.append(df)

    if not frames:
        raise ValueError(f'No usable sheets found in {path}')

    #outer-join all appendices together on soc6
    felten = frames[0]
    for f in frames[1:]:
        felten = felten.merge(f, on='soc6', how='outer')

    print(f'\nFelten combined: {len(felten):,} SOC codes | {felten.shape[1]-1} score columns')
    return felten


felten = load_felten_appendix(FELTEN_FILE)

felten.head(3)

  Sheet "Appendix A": 774 SOC codes | 1 score columns
Skipping sheet "Appendix B": cannot find SOC column. Columns: ['naics', 'industry_title', 'aiie']
  Sheet "Appendix C": 3,271 SOC codes | 2 score columns
Skipping sheet "Appendix D": cannot find SOC column. Columns: ['unnamed:_0', 'abstract_strategy_games', 'real-time_video_games', 'image_recognition', 'visual_question_answering', 'generating_images', 'reading_comprehension', 'language_modeling', 'translation', 'speech_recognition', 'instrumental_track_recognition']
Skipping sheet "Appendix E": cannot find SOC column. Columns: ['o*net_abilities', 'ability-level_ai_exposure']

Felten combined: 4,045 SOC codes | 3 score columns


,soc6,aioe_appendix_a_aioe,aioe_appendix_c_geographic_area,aioe_appendix_c_aige
0,00-1000,NaN,NaN,0.5171
1,00-1001,NaN,NaN,-0.7005
2,00-1003,NaN,NaN,-0.6358


In [19]:
#coverage before merge
print('Pre-merge SOC coverage')
print(f'  BLS (any vintage) : {bls["soc6"].nunique():>5,}')
print(f'  O*NET             : {onet["soc6"].nunique():>5,}')
print(f'  Felten AIOE       : {felten["soc6"].nunique():>5,}')

#triple inner join based on SOC code
master = (
    bls
    .merge(onet,   on='soc6', how='inner', suffixes=('', '_onet'))
    .merge(felten, on='soc6', how='inner', suffixes=('', '_felten'))
)

#move soc code and title forward
id_cols   = ['soc6', 'occ_title']
other_cols = [c for c in master.columns if c not in id_cols]
master = master[id_cols + other_cols]

print(f'\nMaster table: {len(master):,} occupations | {master.shape[1]:,} columns')
print(f'  BLS columns  : {sum(1 for c in master.columns if any(str(y) in c for y in [2019,2022,2024]))}')
print(f'  O*NET columns: {sum(1 for c in master.columns if c.startswith(("onet_","ability","workact")))}' )
print(f'  Felten cols  : {sum(1 for c in master.columns if c.startswith(("aioe_","genai_")))}')

Pre-merge SOC coverage
  BLS (any vintage) :   861
  O*NET             :   774
  Felten AIOE       : 4,045

Master table: 780 occupations | 130 columns
  BLS columns  : 30
  O*NET columns: 95
  Felten cols  : 3


In [20]:
#Duplicate SOC codes
dupe_mask = master['soc6'].duplicated(keep=False)
n_dupes = dupe_mask.sum()
if n_dupes > 0:
    print(f'WARNING: {n_dupes} duplicate soc6 values — inspect below:')
    display(master[dupe_mask][['soc6', 'occ_title']].sort_values('soc6'))
else:
    print('✓ No duplicate SOC codes.')

#missingness
miss = (master.isnull().mean() * 100).round(2).sort_values(ascending=False)
miss_nonzero = miss[miss > 0]
print(f'Columns with any missing data: {len(miss_nonzero)}')
miss_nonzero.head(20)

#coverage diagram
bls_set    = set(bls['soc6'])
onet_set   = set(onet['soc6'])
felten_set = set(felten['soc6'])
master_set = set(master['soc6'])

print('SOC code set sizes')
print(f'  BLS           : {len(bls_set):,}')
print(f'  O*NET         : {len(onet_set):,}')
print(f'  Felten        : {len(felten_set):,}')
print(f'  BLS ∩ O*NET   : {len(bls_set & onet_set):,}')
print(f'  BLS ∩ Felten  : {len(bls_set & felten_set):,}')
print(f'  O*NET ∩ Felten: {len(onet_set & felten_set):,}')
print(f'  Triple ∩ (=Master): {len(master_set):,}')

#SOC codes dropped by each join
not_in_onet   = bls_set - onet_set
not_in_felten = (bls_set & onet_set) - felten_set

print(f'BLS occupations missing from O*NET   : {len(not_in_onet):,}')
print(f'BLS∩O*NET occupations missing from Felten: {len(not_in_felten):,}')

if not_in_onet:
    print('\nSample of SOC codes in BLS but not O*NET:')
    print(sorted(not_in_onet)[:10])

#check key wage fields
wage_check_cols = [c for c in master.columns if 'a_mean' in c or 'a_median' in c]
display(master[['soc6','occ_title'] + wage_check_cols].describe())

,soc6,occ_title
0,11-1011,Chief Executives
1,11-1011,Chief Executives
7,11-3031,Financial Managers
8,11-3031,Financial Managers
9,11-3031,Financial Managers
...,...,...
762,53-6051,Transportation Inspectors
761,53-6051,Transportation Inspectors
763,53-6051,Transportation Inspectors
771,53-7062,"Laborers and Freight, Stock, and Material Move..."


Columns with any missing data: 30
SOC code set sizes
  BLS           : 861
  O*NET         : 774
  Felten        : 4,045
  BLS ∩ O*NET   : 750
  BLS ∩ Felten  : 674
  O*NET ∩ Felten: 682
  Triple ∩ (=Master): 668
BLS occupations missing from O*NET   : 111
BLS∩O*NET occupations missing from Felten: 82

Sample of SOC codes in BLS but not O*NET:
['11-1031', '11-2030', '11-2032', '11-3010', '11-9039', '11-9198', '13-1020', '13-1082', '13-1198', '13-2020']


,a_mean_2019,a_median_2019,a_mean_2022,a_median_2022,a_mean_2024,a_median_2024
count,751.0000,748.0000,776.0000,775.0000,777.0000,774.0000
mean,"64,106.9907","58,747.9947","71,266.0438","64,643.3806","77,281.2870","70,127.5581"
std,"31,517.8578","27,335.9749","35,690.0968","29,475.2009","37,950.5226","31,240.6371"
min,"22,910.0000","21,260.0000","27,870.0000","27,270.0000","30,830.0000","30,160.0000"
25%,"40,010.0000","37,667.5000","45,547.5000","42,630.0000","50,190.0000","47,022.5000"
50%,"56,530.0000","53,005.0000","62,170.0000","58,260.0000","67,580.0000","61,820.0000"
75%,"80,820.0000","74,367.5000","88,960.0000","80,715.0000","95,840.0000","84,255.0000"
max,"237,570.0000","184,460.0000","358,080.0000","211,790.0000","360,240.0000","226,600.0000"


In [21]:
OUT_EXCEL  = PROCESSED / 'master_occupation_table.xlsx'
OUT_PARQUET = PROCESSED / 'master_occupation_table.parquet'

#excel (two sheets: full table + data dict)
with pd.ExcelWriter(OUT_EXCEL, engine='openpyxl') as writer:
    master.to_excel(writer, sheet_name='master', index=False)

    #data dict
    def col_source(col):
        if col in ('soc6', 'occ_title'):        return 'BLS'
        if any(str(y) in col for y in [2019,2022,2024]): return 'BLS OEWS'
        if col.startswith(('ability__','workact__')): return 'O*NET 30.1'
        if col.startswith('onet_'):             return 'O*NET 30.1'
        if col.startswith('aioe_'):             return 'Felten AIOE main'
        if col.startswith('genai_'):            return 'Felten AIOE GenAI'
        return 'unknown'

    dd = pd.DataFrame({
        'column'      : master.columns,
        'source'      : [col_source(c) for c in master.columns],
        'dtype'       : [str(master[c].dtype) for c in master.columns],
        'pct_missing' : [round(master[c].isnull().mean()*100, 2) for c in master.columns],
        'sample_value': [master[c].dropna().iloc[0] if master[c].notna().any() else '' for c in master.columns],
    })
    dd.to_excel(writer, sheet_name='data_dictionary', index=False)

#parquet (could be used for modeling)
master.to_parquet(OUT_PARQUET, index=False)

print(f'Saved Excel  → {OUT_EXCEL}')
print(f'Saved Parquet → {OUT_PARQUET}')
print(f'\nFinal dimensions: {master.shape[0]:,} rows × {master.shape[1]:,} columns')
master[['soc6','occ_title']].head(10)

Saved Excel  → data\processed\master_occupation_table.xlsx
Saved Parquet → data\processed\master_occupation_table.parquet

Final dimensions: 780 rows × 130 columns


,soc6,occ_title
0,11-1011,Chief Executives
1,11-1011,Chief Executives
2,11-1021,General and Operations Managers
3,11-2011,Advertising and Promotions Managers
4,11-2021,Marketing Managers
5,11-2022,Sales Managers
6,11-3021,Computer and Information Systems Managers
7,11-3031,Financial Managers
8,11-3031,Financial Managers
9,11-3031,Financial Managers
